# 02 · 취약성 분석

1. 중심성 (연결·매개·근접·고유벡터)
2. **역(노드)** 전수 제거 → 핵심 역사
3. **구간(엣지)** 전수 제거 → 취약 구간
4. 표적 공격 vs 무작위 제거 → 복원력 곡선
5. 환승역 표적 마비

전수 스윕이 포함되어 전체 실행에 수 분이 걸린다.

In [ ]:
# 저장소 루트에서 실행되도록 경로 이동 (notebooks/ 안에서 열었을 때 대비)
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('작업 경로:', os.getcwd())

In [ ]:
import analyze, networkx as nx, pandas as pd
G = analyze.load_graph()
metro = sorted(max(nx.connected_components(G), key=len))  # 최대 연결요소 = 수도권
print('수도권', len(metro), '개 역')

## 중심성

In [ ]:
cen = analyze.centralities(G, metro)
cen.nlargest(10, '매개중심성')[['역사명','노선명','매개중심성','연결중심성']]

## 역(노드) 제거 전수 스윕 → 핵심 역사

791개 역을 하나씩 제거하며 효율·연결성 저하를 실측한다. **수 분 소요.**

효율은 원 네트워크 크기로 정규화한다(제거된 역은 도달 불가로 0 기여).
남은 노드로 재정규화하면 종단역 제거 시 효율이 '증가'하는 artifact가 생긴다.

In [ ]:
imp, base = analyze.single_removal_sweep(G, metro)
print('단절유발역:', int(imp['분리유발'].sum()), '/', len(imp))
imp.head(10)[['역사명','노선명','효율저하율_%','승객가중효율저하율_%','분리유발']]

## 구간(엣지) 제거 전수 스윕 → 취약 구간

역이 아니라 **선로 구간**이 끊기는 상황(사고·공사)에 대응한다. 역시 수 분 소요.

In [ ]:
eimp = analyze.edge_removal_sweep(G, metro)
print('단절유발 구간:', int(eimp['단절유발'].sum()), '/', len(eimp))
eimp.head(10)[['역A','역B','구간유형','효율저하율_%','승객가중효율저하율_%','단절유발']]

In [ ]:
# 구간 매개중심성 상위 — 통과 통행량이 몰리는 병목 구간
eimp.nlargest(8, '구간매개중심성')[['역A','역B','구간유형','구간매개중심성']]

## 복원력 곡선 (표적 공격 vs 무작위 제거)

In [ ]:
net = analyze.Net(G, metro)
curves = {}
for st, runs in [('random', 10), ('degree', 1), ('betweenness', 1), ('adaptive', 1)]:
    curves[st] = analyze.removal_curve(G, metro, st, frac=0.25, step=4,
                                       runs=runs, net=net)
    d = curves[st]; a = d[d['제거비율'] >= 0.10].head(1)
    print(f"{st:12s} 10% 제거 → 효율 {a['효율비율'].iloc[0]*100:.1f}% "
          f"/ LCC {a['LCC비율'].iloc[0]*100:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
for c in ['Malgun Gothic','AppleGothic','NanumGothic','Noto Sans CJK KR']:
    if any(c in f.name for f in font_manager.fontManager.ttflist):
        plt.rcParams['font.family'] = c; break
plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(7, 4.2))
for st, lab in [('random','무작위'), ('degree','연결중심성'),
                ('betweenness','매개중심성'), ('adaptive','적응형')]:
    d = curves[st]
    ax.plot(d['제거비율']*100, d['효율비율']*100, lw=2, label=lab)
ax.set_xlabel('제거된 역사 비율 (%)'); ax.set_ylabel('전역 효율 (%)')
ax.legend(); ax.grid(alpha=.4); plt.show()

## 전체 파이프라인 한 번에 실행

위 단계 + 환승역 표적 마비 + 대전 사례까지 실행해 `results/`에 CSV로 저장한다.

In [ ]:
analyze.main()